# Setup

In [ ]:
import os
os.environ["HUGGINGFACE_API"] = ""
os.environ["GIT_TOKEN"] = ""

In [ ]:
import os
from huggingface_hub import login


huggingface_api = os.environ["HUGGINGFACE_API"]
git_token = os.environ["GIT_TOKEN"]

if huggingface_api is None:
    raise RuntimeError("❌ Missing HUGGINGFACE_API in .env")

login(token=huggingface_api)


In [ ]:
!git clone https://{git_token}@github.com/BGKhanh/Reasoning-Techniques-on-LLM.git

### Uncomment and run when use default template like Pytorch (vastai)

In [ ]:
# %cd /workspace/Reasoning-Techniques-on-LLM

In [ ]:
# !git pull

In [ ]:
# # Add --ignore-installed if PyJWT error occurs or use "sudo apt remove python3-jwt"
# !python3 -m pip install -Uqr requirements.txt --ignore-installed

In [ ]:
# !export FLASH_ATTENTION_SKIP_CUDA_BUILD=TRUE

In [ ]:
# !python3 -m pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl

In [ ]:
# %cd lm-evaluation-harness
# !python3 -m pip install -qe ."[hf,api,vllm]"

### Used with the specific docker images

In [ ]:
# %cd /workspace/Reasoning-Techniques-on-LLM
# !git pull

In [ ]:
## Access this https://github.com/mjun0812/flash-attention-prebuild-wheels for most suitable version of Flash Attention
# !export FLASH_ATTENTION_SKIP_CUDA_BUILD=TRUE
# !python3 -m pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl

In [ ]:
# %cd lm-evaluation-harness
# !python3 -m pip install -qe ."[hf,api,vllm]"

# Simple inference

In [ ]:
# Add tensor_parallel_size == number of GPUs, gpu_memory_utilization usually should be higher than 0.9 (like 0.975)
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model vllm \
  --model_args pretrained=google/gemma-4-E2B-it,max_model_len=16384,gpu_memory_utilization=0.95 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --batch_size 32 \
  --output_path results/lm_eval/gemma_few_shot_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":0}' \
  --confirm_run_unsafe_code

# Host VLLM

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

def _find_project_root(start: Path, max_up: int = 5) -> Path:
    cur = start.resolve()
    for _ in range(max_up):
        if (cur / "src" / "prompt_templates").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start  

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "nvidia/Gemma-4-26B-A4B-NVFP4")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))

BASE_URL = os.getenv("BASE_URL", f"http://127.0.0.1:{VLLM_PORT}/v1")
base_url_completions = BASE_URL.rstrip("/") + "/completions"

OUTPUT_DIR = PROJECT_ROOT / "results" / "lm_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Khởi chạy vLLM OpenAI server cục bộ cho student LM (chạy sau cell Configuration).
# Tùy VRAM có thể thêm vào _vllm_cmd: --max-model-len 4096 --gpu-memory-utilization 0.9 --dtype bfloat16
import subprocess
import sys
import time
import urllib.error
import urllib.request

if "_vllm_proc" in globals() and _vllm_proc is not None and _vllm_proc.poll() is None:
    _vllm_proc.terminate()
    try:
        _vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _vllm_proc.kill()

_vllm_env = os.environ.copy()
_hf = (
    _vllm_env.get("HUGGINGFACE_API_KEY")
    or _vllm_env.get("HF_TOKEN")
    or _vllm_env.get("HUGGINGFACE_HUB_TOKEN")
    or _vllm_env.get("HUGGINGFACE_API", "")
)
if _hf:
    _vllm_env.setdefault("HF_TOKEN", _hf)
    _vllm_env.setdefault("HUGGINGFACE_HUB_TOKEN", _hf)

_vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", VLLM_MODEL_ID,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    # "--dtype", "bfloat16",
    "--gpu-memory-utilization", "0.975",
    # "--tensor-parallel-size", "2",
    "--max-model-len", "16384",
    # "--max-num-batched-tokens", "8192"
]

_vllm_log = OUTPUT_DIR / "vllm_server.log"
_flog = open(_vllm_log, "a", encoding="utf-8", buffering=1)
_flog.write(f"\n\n==== vLLM start {time.strftime('%Y-%m-%d %H:%M:%S')} ====\n")
_flog.write(" ".join(_vllm_cmd) + "\n")
_flog.flush()

print("Starting vLLM — log:", _vllm_log)
_vllm_proc = subprocess.Popen(
    _vllm_cmd,
    env=_vllm_env,
    stdout=_flog,
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

_health = BASE_URL.rstrip("/") + "/models"
_deadline = time.time() + float(os.getenv("VLLM_START_TIMEOUT_S", "1200"))
_last_err = None
while time.time() < _deadline:
    if _vllm_proc.poll() is not None:
        _flog.close()
        tail = _vllm_log.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"vLLM exited early (code={_vllm_proc.returncode}). See {_vllm_log}. Tail:\n{tail}"
        )
    try:
        with urllib.request.urlopen(_health, timeout=5) as r:
            if r.status == 200:
                print(f"vLLM ready: {_health}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        _last_err = e
    time.sleep(2.0)
else:
    if _vllm_proc.poll() is None:
        _vllm_proc.terminate()
    _flog.close()
    raise RuntimeError(f"vLLM did not become ready in time. Last error: {_last_err}. Log: {_vllm_log}")

In [ ]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model local-completions \
  --model_args model=nvidia/Gemma-4-26B-A4B-NVFP4,base_url=http://127.0.0.1:8000/v1/completions,num_concurrent=32,max_length=16384 \
  --tasks tasks/vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_few_shot_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}' \
  --confirm_run_unsafe_code

# Host llama.cpp

In [ ]:
# Uncomment and run when use default template like Pytorch (vastai) 


# !git clone --depth 1 https://github.com/ggml-org/llama.cpp /workspace/llama.cpp
# !cd /workspace/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -50
# !cd /workspace/llama.cpp && cmake --build build --config Release -j $(nproc) 2>&1 | tail -80
# !ls -la /workspace/llama.cpp/build/bin/llama-server

In [ ]:
import subprocess
import sys
import time
import urllib.request
import urllib.error
import os
import atexit
from pathlib import Path

# ==========================================
# 0. CẤU HÌNH
# ==========================================
# Đường dẫn binary llama-server đã build (xem lại bước build cmake ở câu trước).
LLAMA_SERVER_BIN = "/workspace/llama.cpp/build/bin/llama-server"

# Repo HF chứa file GGUF — HÃY TỰ KIỂM TRA repo này tồn tại thật trên huggingface.co
# trước khi chạy (tôi không browse được để xác nhận tên repo/tên file chính xác).
HF_REPO_ID = "google/gemma-4-26B-A4B-it-qat-q4_0-gguf"
HF_FILENAME = "gemma-4-26B_q4_0-it.gguf"  # đổi đúng tên file thật trong repo

PORT = 8000
N_CTX = 16384      # tổng context budget, CHIA cho các slot bên dưới
N_PARALLEL = 4     # số slot song song — tune theo VRAM còn trống
# Có nghĩa là 1 slot sẽ có max_lenght=N_CTX/N_PARALLEL (vd: N_CTX = 16384, N_PARALLEL = 4 -> max_length 1 slot = 4096)
# nếu N_PARRALLEL tăng phải tăng N_CTX để đảm bảo không bị maximmum context exceed.
LOG_FILE = Path("llama_server.log")

if not Path(LLAMA_SERVER_BIN).exists():
    raise FileNotFoundError(
        f"Không tìm thấy llama-server tại: {LLAMA_SERVER_BIN}. "
        "Hãy build trước bằng cmake (xem hướng dẫn build ở các bước trước)."
    )

# ==========================================
# 1. TẢI MODEL GGUF VỀ LOCAL (nếu chưa có)
# ==========================================
# llama-server (binary gốc) KHÔNG có cờ --hf_model_repo_id kiểu llama_cpp.server.
# Để kiểm soát chắc chắn tên file, tải tường minh bằng huggingface_hub trước.
from huggingface_hub import hf_hub_download

hf_token = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACE_API_KEY")
)

print(f"⬇️  Đang kiểm tra/tải model từ {HF_REPO_ID} ...")
MODEL_PATH = Path(
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        local_dir=".",
        token=hf_token,
    )
)
print(f"✅ Model sẵn sàng tại: {MODEL_PATH.resolve()}")

# ==========================================
# 2. HÀM DỌN DẸP TIẾN TRÌNH TỰ ĐỘNG
# ==========================================
def cleanup_llama_server():
    global _llama_proc
    if "_llama_proc" in globals() and _llama_proc is not None:
        if _llama_proc.poll() is None:
            print("\n[Cleanup] Đang tắt llama.cpp server an toàn...")
            _llama_proc.terminate()
            try:
                _llama_proc.wait(timeout=5)
                print("[Cleanup] Đã tắt server thành công.")
            except subprocess.TimeoutExpired:
                print("[Cleanup] Server không phản hồi, đang ép buộc tắt (kill)...")
                _llama_proc.kill()


atexit.register(cleanup_llama_server)
cleanup_llama_server()  # dọn dẹp nếu chạy lại cell nhiều lần

# ==========================================
# 3. KHỞI CHẠY llama-server VỚI CONTINUOUS BATCHING
# ==========================================
llama_cmd = [
    LLAMA_SERVER_BIN,
    "-m", str(MODEL_PATH),
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "-c", str(N_CTX),       # tổng context, chia cho các slot
    "-ngl", "99",            # offload toàn bộ layer lên GPU
    "-np", str(N_PARALLEL),  # số slot xử lý song song -> tăng throughput thật
    "-cb",                   # continuous batching (mặc định đã bật, ghi rõ cho chắc)
]

flog = open(LOG_FILE, "w", encoding="utf-8", buffering=1)
flog.write(f"==== Khởi chạy lúc {time.strftime('%H:%M:%S')} ====\n{' '.join(llama_cmd)}\n\n")
flog.flush()

print(f"🚀 Đang khởi chạy llama-server (Log: {LOG_FILE})...")
_llama_proc = subprocess.Popen(
    llama_cmd, stdout=flog, stderr=subprocess.STDOUT, env=os.environ.copy()
)

# ==========================================
# 4. HEALTH CHECK
# ==========================================
# Binary gốc CÓ endpoint /health (khác llama_cpp.server chỉ có /v1/models).
health_url = f"http://127.0.0.1:{PORT}/health"
deadline = time.time() + 300  # model lớn load lâu hơn, nới rộng timeout
last_err = None

print("⏳ Đang chờ model nạp vào GPU (model lớn có thể mất vài phút)...")

while time.time() < deadline:
    if _llama_proc.poll() is not None:
        flog.close()
        tail = LOG_FILE.read_text(encoding="utf-8", errors="replace")[-2000:]
        raise RuntimeError(f"❌ Server bị crash (Exit code: {_llama_proc.returncode}). Log:\n{tail}")
    try:
        with urllib.request.urlopen(health_url, timeout=2) as r:
            if r.status == 200:
                print(f"✅ Sẵn sàng tại: http://127.0.0.1:{PORT} (n_parallel={N_PARALLEL}, n_ctx={N_CTX})")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        last_err = e
    time.sleep(2.0)
else:
    cleanup_llama_server()
    flog.close()
    raise RuntimeError(f"⏳ Quá thời gian chờ. Lỗi cuối: {last_err}. Log: {LOG_FILE}")

In [ ]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model local-chat-completions \
  --model_args model=google/gemma-4-26B-A4B-it-qat-q4_0-gguf,base_url=http://127.0.0.1:8000/v1/chat/completions,num_concurrent=4,max_retries=3,timeout=300,max_length=16384 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_gguf_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}' \
  --confirm_run_unsafe_code